# Comprehensions => Dict, Set & Generator Expressions

Set and dict comprehensions build collections. Generator expressions build nothing until you ask.

| Kind | Syntax | Notes |
|---|---|---|
| Set comprehension | `{x for x in it}` | Unique items, no order |
| Dict comprehension | `{k: v for k, v in it}` | The last duplicate key wins |
| Generator expression | `(x for x in it)` | Lazy, single use |
| Sole argument | `sum(x for x in it)` | No extra parentheses needed |
| Empty set | `set()` | `{}` is an empty dict |

---

## Set Comprehension

```python
{expression for item in iterable if condition}
```

- The result is a `set`: **unique** items, **no guaranteed order**.
- Use it to remove duplicates while transforming.

```python
{word[0] for word in words}      # the set of first letters
```

**Watch out:** `{}` is an empty **dict**, not an empty set. Use `set()` for an empty set.

## Dict Comprehension

```python
{key_expression: value_expression for item in iterable if condition}
```

- Keys must be **hashable** and unique.
- If the same key appears more than once, the **last value wins**.

### Common Patterns

| Pattern | Example |
|---|---|
| Build from two lists | `{k: v for k, v in zip(keys, values)}` |
| Transform values | `{k: v * 2 for k, v in d.items()}` |
| Filter items | `{k: v for k, v in d.items() if v > 0}` |
| Swap keys and values | `{v: k for k, v in d.items()}` (values must be unique) |
| Lookup table from records | `{r["id"]: r for r in records}` |

## Generator Expression

```python
(expression for item in iterable if condition)
```

- It does **not** build a collection. It returns a **generator** that produces one value at a time.
- It is **lazy**: nothing is computed until a value is requested.
- It is **single use**: once exhausted, it stays empty.

### Passing It to a Function

When a generator expression is the **only** argument, the extra parentheses can be dropped:

```python
sum(n * n for n in range(10))
max(len(w) for w in words)
", ".join(w.upper() for w in words)
```

With other arguments, keep the parentheses:

```python
sorted((w for w in words if "e" in w), key=len)
```

### Short-Circuiting

`any()` and `all()` stop as soon as the answer is known. With a generator expression they do not evaluate the remaining items.

## List vs Generator Expression

| | List comprehension | Generator expression |
|---|---|---|
| Brackets | `[ ]` | `( )` |
| Builds everything now | Yes | No, on demand |
| Memory | Stores every item | Stores only its state |
| Reusable | Yes | No, single use |
| `len()`, indexing | Yes | No |
| Best for | Results you keep or reuse | Results you consume once |

## Which Iterable Is Evaluated First?

The **leftmost** `for` iterable is evaluated immediately, when the generator is created. Everything else is evaluated later, during iteration.

```python
(x for x in undefined_name)                       # NameError right away
(x for x in range(3) for y in undefined_name)     # no error until iterated
```

## Key Rules

- `{}` is a dict. Use `set()` for an empty set.
- Dict comprehension keys must be hashable. Duplicate keys keep the last value.
- Use a generator expression when you consume the values once.
- Use a list comprehension when you need `len()`, indexing, or to iterate more than once.

## Source

https://docs.python.org/3/tutorial/datastructures.html#sets

https://docs.python.org/3/tutorial/datastructures.html#dictionaries

https://docs.python.org/3/reference/expressions.html

In [ ]:
words = ["apple", "banana", "avocado", "blueberry", "cherry"]

# Set comprehension
print(sorted({w[0] for w in words}))                        # first letters, no duplicates
print(sorted({n % 3 for n in range(10)}))
print(type({}).__name__, type({n for n in range(3)}).__name__)   # {} is a dict, not a set

# Dict comprehension
names = ["ann", "bob", "cy"]
ages = [30, 25, 41]
people = {name: age for name, age in zip(names, ages)}
print(people)
print({name: age + 1 for name, age in people.items()})           # transform values
print({name: age for name, age in people.items() if age > 26})   # filter items
print({age: name for name, age in people.items()})               # swap (values must be unique)
print({w: len(w) for w in words})
print({n % 3: n for n in range(10)})                             # duplicate keys: last value wins

# Lookup table from records
records = [{"id": 1, "name": "ann"}, {"id": 2, "name": "bob"}]
by_id = {r["id"]: r for r in records}
print(by_id[2]["name"])

# Generator expression
squares = (n * n for n in range(5))
print(type(squares).__name__)
print(next(squares), next(squares))         # values are produced on demand
print(list(squares))                        # the remaining values
print(list(squares))                        # exhausted: single use -> []

# A sole generator argument needs no extra parentheses
print(sum(n * n for n in range(10)))
print(max(len(w) for w in words))
print(", ".join(w.upper() for w in words))
print(sorted((w for w in words if "e" in w), key=len))      # extra argument: keep parentheses

# Short-circuiting with any()
checked = []

def is_big(n):
    checked.append(n)
    return n > 2

print(any(is_big(n) for n in range(10)))    # generator: stops at the first True
print(checked)

checked.clear()
print(any([is_big(n) for n in range(10)]))  # list: evaluates everything first
print(len(checked))

# Memory: list vs generator
import sys
big_list = [n for n in range(100_000)]
big_gen = (n for n in range(100_000))
print(sys.getsizeof(big_list) > sys.getsizeof(big_gen))

# Only the FIRST iterable is evaluated immediately
try:
    (x for x in undefined_name)
except NameError as error:
    print("eager:", type(error).__name__)

lazy = (x for x in range(3) for y in undefined_name)     # no error yet
try:
    next(lazy)
except NameError as error:
    print("lazy:", type(error).__name__)